https://chatgpt.com/share/6a5b8495-525c-83e8-833c-2c8d567d561a

#**Step 1: Install Dependencies**

In [ ]:
# Core LLM
!pip install -q google-generativeai

# LangChain
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q langchain-google-genai

# Vector Database
!pip install -q chromadb

# Embeddings
!pip install -q sentence-transformers

# Document Loaders
!pip install -q pypdf
!pip install -q python-docx
!pip install -q docx2txt
!pip install -q openpyxl
!pip install -q unstructured

# Utilities
!pip install -q pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

#**Step 2: Import Libraries**

In [ ]:
import os
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    CSVLoader,
    UnstructuredExcelLoader,
    UnstructuredPowerPointLoader,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

import google.generativeai as genai

import pandas as pd

/tmp/ipykernel_1304/1537980274.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


#**Step 3: Configuration**

In [ ]:
# ===========================
# Project Configuration
# ===========================

GOOGLE_API_KEY = "####################" #paste your Google API key here

DATA_FOLDER = "/content/data"

VECTOR_DB_PATH = "/content/chroma_db"

COLLECTION_NAME = "enterprise_documents"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

CHUNK_SIZE = 1000

CHUNK_OVERLAP = 200

TOP_K = 5

FETCH_K = 20

TEMPERATURE = 0

Step 4: Configure Gemini

In [ ]:
genai.configure(api_key=GOOGLE_API_KEY)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=TEMPERATURE,
)

Step 5: Create Data Folder

In [ ]:
os.makedirs(DATA_FOLDER, exist_ok=True)

print("Upload all documents into:")

print(DATA_FOLDER)

Upload all documents into:
/content/data


Step 6: Universal Document Loader

In [ ]:
# ===========================
# Universal Document Loader
# ===========================

LOADER_MAPPING = {
    ".pdf": PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".txt": TextLoader,
    ".md": TextLoader,
    ".csv": CSVLoader,
    ".xlsx": UnstructuredExcelLoader,
    ".pptx": UnstructuredPowerPointLoader,
}

Step 7: Load Documents Function

In [ ]:
def load_documents(data_folder: str):
    """
    Load all supported documents from the specified folder.
    """

    documents = []

    data_path = Path(data_folder)

    for file_path in data_path.iterdir():

        if not file_path.is_file():
            continue

        extension = file_path.suffix.lower()

        if extension not in LOADER_MAPPING:
            print(f"❌ Skipped: {file_path.name}")
            continue

        try:

            loader = LOADER_MAPPING[extension](str(file_path))

            docs = loader.load()

            documents.extend(docs)

            print(f"✅ Loaded: {file_path.name}")

        except Exception as e:

            print(f"⚠ Error loading {file_path.name}")

            print(e)

    return documents

Step 8: Load All Documents

In [ ]:
documents = load_documents(DATA_FOLDER)

print(f"\nTotal Documents Loaded : {len(documents)}")

✅ Loaded: JD_Business Intelligence (Intern).pdf
✅ Loaded: Capstone_Machine_Unlearning.pdf

Total Documents Loaded : 7


Step 9: Preview Documents

In [ ]:
for i, doc in enumerate(documents[:3]):

    print("=" * 80)

    print(f"Document : {i+1}")

    print("Metadata")

    print(doc.metadata)

    print("\nContent")

    print(doc.page_content[:500])

Document : 1
Metadata
{'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-07-15T13:33:20+05:30', 'author': 'Yedilat Peguero', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-07-15T13:33:20+05:30', 'sourcemodified': "D:20260715133320+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/JD_Business Intelligence (Intern).pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

Content
Job Description:
Business Intelligence Intern
Guardian (Guardian Life Insurance Company of America) is on a
transformational journey to evolve into a forward-thinking mutual insurance
company committed to championing the well-being of its customers, colleagues
and communities.
Guardian is seeking smart Interns with the zeal to build innovative BI solutions. In
this role as Business Intelligence Intern, you'll be working in a team of Data
Analyst led by Lead Business Intelligence to develop advan
Document : 2
Metadata
{'producer': '', 'creator': 'WPS Docs'

Step 10: Text Cleaning Function

In [ ]:
# ===========================
# Text Cleaning
# ===========================

import re

def clean_text(text: str) -> str:
    """
    Clean extracted text while preserving meaning.
    """

    if not text:
        return ""

    # Remove extra spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

Step 11: Preprocess Documents

In [ ]:
# ===========================
# Preprocess Documents
# ===========================

from langchain_core.documents import Document

def preprocess_documents(documents):
    """
    Clean documents while preserving metadata.
    """

    cleaned_documents = []

    for doc in documents:

        cleaned_text = clean_text(doc.page_content)

        if not cleaned_text:
            continue

        cleaned_documents.append(
            Document(
                page_content=cleaned_text,
                metadata=doc.metadata
            )
        )

    return cleaned_documents

Step 12: Clean Documents

In [ ]:
documents = preprocess_documents(documents)

print(f"Total Clean Documents : {len(documents)}")

Total Clean Documents : 7


Step 13: Preview Cleaned Document

In [ ]:
print("=" * 80)

print(documents[0].metadata)

print()

print(documents[0].page_content[:1000])

{'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-07-15T13:33:20+05:30', 'author': 'Yedilat Peguero', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-07-15T13:33:20+05:30', 'sourcemodified': "D:20260715133320+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/JD_Business Intelligence (Intern).pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

Job Description:
Business Intelligence Intern
Guardian (Guardian Life Insurance Company of America) is on a
transformational journey to evolve into a forward-thinking mutual insurance
company committed to championing the well-being of its customers, colleagues
and communities.
Guardian is seeking smart Interns with the zeal to build innovative BI solutions. In
this role as Business Intelligence Intern, you'll be working in a team of Data
Analyst led by Lead Business Intelligence to develop advanced data solutions, to
drive enterprise-wide innovation across various business lines and G

Step 14: Remove Duplicate Documents (Recommended)

In [ ]:
# ===========================
# Remove Duplicate Documents
# ===========================

def remove_duplicate_documents(documents):
    """
    Remove duplicate document contents.
    """

    unique_docs = []
    seen = set()

    for doc in documents:

        content = doc.page_content.strip()

        if content in seen:
            continue

        seen.add(content)
        unique_docs.append(doc)

    return unique_docs

Step 15: Apply Duplicate Removal

In [ ]:
documents = remove_duplicate_documents(documents)

print(f"Documents After Removing Duplicates : {len(documents)}")

Documents After Removing Duplicates : 7


Step 16: Document Statistics (Useful for Debugging)

In [ ]:
total_chars = sum(len(doc.page_content) for doc in documents)

avg_chars = total_chars / len(documents)

print(f"Documents       : {len(documents)}")
print(f"Total Characters: {total_chars:,}")
print(f"Average Length  : {avg_chars:.2f}")

Documents       : 7
Total Characters: 9,750
Average Length  : 1392.86


Step 17: Configure Text Splitter

In [ ]:
# ===========================
# Text Splitter
# ===========================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

Step 18: Split Documents into Chunks

In [ ]:
# ===========================
# Split Documents
# ===========================

chunked_documents = text_splitter.split_documents(documents)

print(f"Total Chunks : {len(chunked_documents)}")

Total Chunks : 14


Step 19: Add Chunk IDs

In [ ]:
# ===========================
# Add Chunk IDs
# ===========================

from collections import defaultdict

chunk_counter = defaultdict(int)

for chunk in chunked_documents:

    source = Path(chunk.metadata.get("source", "unknown")).stem

    page = chunk.metadata.get("page", 0)

    key = f"{source}_page_{page}"

    chunk_counter[key] += 1

    chunk.metadata["chunk_id"] = (
        f"{source}_p{page}_c{chunk_counter[key]}"
    )

Step 20: Preview Chunks

In [ ]:
for i, chunk in enumerate(chunked_documents[:3]):

    print("=" * 80)

    print(f"Chunk {i+1}")

    print()

    print(chunk.metadata)

    print()

    print(chunk.page_content[:600])

Chunk 1

{'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-07-15T13:33:20+05:30', 'author': 'Yedilat Peguero', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-07-15T13:33:20+05:30', 'sourcemodified': "D:20260715133320+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/data/JD_Business Intelligence (Intern).pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'chunk_id': 'JD_Business Intelligence (Intern)_p0_c1'}

Job Description:
Business Intelligence Intern
Guardian (Guardian Life Insurance Company of America) is on a
transformational journey to evolve into a forward-thinking mutual insurance
company committed to championing the well-being of its customers, colleagues
and communities.
Guardian is seeking smart Interns with the zeal to build innovative BI solutions. In
this role as Business Intelligence Intern, you'll be working in a team of Data
Analyst led by Lead Business Intelligence to develop advanced data solutions, to
driv

Step 21: Chunk Statistics

In [ ]:
chunk_lengths = [len(doc.page_content) for doc in chunked_documents]

print(f"Total Chunks      : {len(chunk_lengths)}")
print(f"Smallest Chunk    : {min(chunk_lengths)}")
print(f"Largest Chunk     : {max(chunk_lengths)}")
print(f"Average Chunk     : {sum(chunk_lengths)/len(chunk_lengths):.2f}")

Total Chunks      : 14
Smallest Chunk    : 228
Largest Chunk     : 995
Average Chunk     : 775.43


Step 22: Verify Metadata

In [ ]:
chunked_documents[0].metadata

# Example output:

# {
#     'source': '/content/data/employee_handbook.pdf',
#     'page': 0,
#     'chunk_id': 'employee_handbook_p0_c1'
# }

{'producer': '',
 'creator': 'WPS Docs',
 'creationdate': '2026-07-15T13:33:20+05:30',
 'author': 'Yedilat Peguero',
 'comments': '',
 'company': '',
 'keywords': '',
 'moddate': '2026-07-15T13:33:20+05:30',
 'sourcemodified': "D:20260715133320+05'30'",
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '/content/data/JD_Business Intelligence (Intern).pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1',
 'chunk_id': 'JD_Business Intelligence (Intern)_p0_c1'}

Step 23: Install Latest Packages

In [ ]:
# Latest LangChain Integrations
!pip install -q langchain-huggingface
!pip install -q langchain-chroma

Step 24: Import Embedding & Chroma

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

Step 25: Create Embedding Model

In [ ]:
# ===========================
# Embedding Model
# ===========================

embedding_function = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 26: Create Persistent ChromaDB

In [ ]:
# ===========================
# Create Vector Database
# ===========================

vectorstore = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_function,
    persist_directory=VECTOR_DB_PATH,
    collection_name=COLLECTION_NAME,
)

Step 27: Verify Database

In [ ]:
print(f"Total Chunks Indexed : {vectorstore._collection.count()}")

Total Chunks Indexed : 14


Step 28: Reload Existing Database

In [ ]:
# ===========================
# Load Existing ChromaDB
# ===========================

vectorstore = Chroma(
    persist_directory=VECTOR_DB_PATH,
    embedding_function=embedding_function,
    collection_name=COLLECTION_NAME,
)

Step 29: Test Similarity Search

In [ ]:
query = "What is the unlearning?"

results = vectorstore.similarity_search(
    query=query,
    k=3
)

for i, doc in enumerate(results, 1):

    print("=" * 80)

    print(f"Result {i}")

    print(doc.metadata)

    print()

    print(doc.page_content[:500])

Result 1
{'company': '', 'comments': '', 'chunk_id': 'Capstone_Machine_Unlearning_p0_c2', 'moddate': '2026-07-17T10:00:25+05:30', 'keywords': '', 'page_label': '1', 'total_pages': 3, 'creationdate': '2026-07-17T10:00:25+05:30', 'sourcemodified': "D:20260717100025+05'30'", 'title': '', 'subject': '', 'page': 0, 'source': '/content/data/Capstone_Machine_Unlearning.pdf', 'trapped': '/False', 'producer': '', 'creator': 'WPS Docs', 'author': 'Gokul U 22MIS0291'}

deletion requests, making them unsuitable for complying with
privacy regulations such as the General Data Protection Regulation
(GDPR) and the Right to be Forgotten. The conventional solution
involves retraining the entire model after removing the requested
records, which is computationally expensive and time-consuming.
This project proposes a Privacy-Preserving Machine Unlearning
Framework using the MIMIC-IV Electronic Health Record
dataset. The proposed system enables selective removal of a
pa
Result 2
{'source': '/content/data/C

Step 30: Similarity Search with Scores

In [ ]:
results = vectorstore.similarity_search_with_score(
    query=query,
    k=3
)

for doc, score in results:

    print("=" * 80)

    print(f"Score : {score:.4f}")

    print(doc.metadata)

    print()

    print(doc.page_content[:500])

Score : 0.7186
{'page_label': '1', 'title': '', 'author': 'Gokul U 22MIS0291', 'keywords': '', 'creationdate': '2026-07-17T10:00:25+05:30', 'trapped': '/False', 'company': '', 'sourcemodified': "D:20260717100025+05'30'", 'page': 0, 'source': '/content/data/Capstone_Machine_Unlearning.pdf', 'subject': '', 'comments': '', 'producer': '', 'chunk_id': 'Capstone_Machine_Unlearning_p0_c2', 'total_pages': 3, 'moddate': '2026-07-17T10:00:25+05:30', 'creator': 'WPS Docs'}

deletion requests, making them unsuitable for complying with
privacy regulations such as the General Data Protection Regulation
(GDPR) and the Right to be Forgotten. The conventional solution
involves retraining the entire model after removing the requested
records, which is computationally expensive and time-consuming.
This project proposes a Privacy-Preserving Machine Unlearning
Framework using the MIMIC-IV Electronic Health Record
dataset. The proposed system enables selective removal of a
pa
Score : 0.7676
{'total_pages':

Step 31: Metadata Filtering

In [ ]:
results = vectorstore.similarity_search(
    query=query,
    k=3,
    filter={
        "source": "/content/data/Capstone_Machine_Unlearning.pdf"
    }
)

for doc in results:

    print(doc.metadata)

    print(doc.page_content[:300])

    print()

{'page_label': '1', 'page': 0, 'producer': '', 'trapped': '/False', 'source': '/content/data/Capstone_Machine_Unlearning.pdf', 'creationdate': '2026-07-17T10:00:25+05:30', 'moddate': '2026-07-17T10:00:25+05:30', 'creator': 'WPS Docs', 'author': 'Gokul U 22MIS0291', 'chunk_id': 'Capstone_Machine_Unlearning_p0_c2', 'keywords': '', 'company': '', 'subject': '', 'comments': '', 'sourcemodified': "D:20260717100025+05'30'", 'title': '', 'total_pages': 3}
deletion requests, making them unsuitable for complying with
privacy regulations such as the General Data Protection Regulation
(GDPR) and the Right to be Forgotten. The conventional solution
involves retraining the entire model after removing the requested
records, which is computationally expensiv

{'comments': '', 'company': '', 'keywords': '', 'producer': '', 'title': '', 'chunk_id': 'Capstone_Machine_Unlearning_p1_c1', 'moddate': '2026-07-17T10:00:25+05:30', 'sourcemodified': "D:20260717100025+05'30'", 'author': 'Gokul U 22MIS0291', 'cr

Step 32: Create Retriever (MMR)

In [ ]:
# ===========================
# MMR Retriever
# ===========================

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K,
        "fetch_k": FETCH_K
    }
)

Step 33: Test Retriever

In [ ]:
question = "What is guardian?"

documents = retriever.invoke(question)

for i, doc in enumerate(documents, 1):

    print("=" * 80)

    print(f"Document {i}")

    print(doc.metadata)

    print()

    print(doc.page_content[:500])

Document 1
{'page_label': '3', 'comments': '', 'subject': '', 'total_pages': 4, 'creator': 'WPS Docs', 'company': '', 'producer': '', 'trapped': '/False', 'author': 'Yedilat Peguero', 'source': '/content/data/JD_Business Intelligence (Intern).pdf', 'keywords': '', 'moddate': '2026-07-15T13:33:20+05:30', 'title': '', 'chunk_id': 'JD_Business Intelligence (Intern)_p2_c3', 'sourcemodified': "D:20260715133320+05'30'", 'creationdate': '2026-07-15T13:33:20+05:30', 'page': 2}

are helping to uplift communities through thoughtful corporate impact
programs. Guardian, which is based in New York City, is a leading provider of
life, disability, dental, and other benefits, and has received accolades for its
Document 2
{'title': '', 'total_pages': 4, 'creator': 'WPS Docs', 'comments': '', 'chunk_id': 'JD_Business Intelligence (Intern)_p2_c1', 'page': 2, 'trapped': '/False', 'sourcemodified': "D:20260715133320+05'30'", 'subject': '', 'company': '', 'creationdate': '2026-07-15T13:33:20+05:30', 'page_l

Step 34: Create VectorStore Function

In [ ]:
# ===========================
# Create / Load ChromaDB
# ===========================

from pathlib import Path

def get_vectorstore(documents=None):
    """
    Create a new ChromaDB if it doesn't exist,
    otherwise load the existing database.
    """

    db_path = Path(VECTOR_DB_PATH)

    # Existing Database
    if db_path.exists() and any(db_path.iterdir()):

        print("Loading existing ChromaDB...")

        return Chroma(
            persist_directory=VECTOR_DB_PATH,
            embedding_function=embedding_function,
            collection_name=COLLECTION_NAME
        )

    # New Database
    print("Creating ChromaDB...")

    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_function,
        persist_directory=VECTOR_DB_PATH,
        collection_name=COLLECTION_NAME
    )

    return vectorstore

Step 35: Initialize VectorStore

In [ ]:
vectorstore = get_vectorstore(chunked_documents)

print(f"Indexed Chunks : {vectorstore._collection.count()}")

Loading existing ChromaDB...
Indexed Chunks : 14


Step 36: Create Retriever Function

In [ ]:
# ===========================
# Retriever
# ===========================

def get_retriever(vectorstore):

    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": TOP_K,
            "fetch_k": FETCH_K
        }
    )

    return retriever

Step 37: Initialize Retriever

In [ ]:
retriever = get_retriever(vectorstore)

Step 38: Prompt Template

In [ ]:
# ===========================
# Prompt Template
# ===========================

SYSTEM_PROMPT = """
You are an AI Enterprise Document Assistant.

Answer the user's question ONLY using the retrieved document context.

Rules:

1. Do not use outside knowledge.

2. If the answer is not found, reply exactly:

"I couldn't find this information in the uploaded documents."

3. Be clear and concise.

4. If multiple documents contain relevant information,
combine them into one answer.

5. Preserve important names, numbers, dates and policies.

6. At the end, include the sources used.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}")
    ]
)

Step 39: Format Retrieved Context

In [ ]:
# ===========================
# Context Formatter
# ===========================

def format_context(documents):

    formatted_context = []

    for doc in documents:

        source = Path(
            doc.metadata.get(
                "source",
                "Unknown"
            )
        ).name

        page = doc.metadata.get(
            "page",
            "N/A"
        )

        formatted_context.append(

f"""
Source : {source}
Page   : {page}

Content:
{doc.page_content}
"""
        )

    return "\n\n".join(formatted_context)

Step 40: Build Prompt Messages

In [ ]:
def build_messages(question):

    documents = retriever.invoke(question)

    context = format_context(documents)

    messages = prompt.format_messages(
        context=context,
        question=question
    )

    return messages, documents

Step 41: Generate Response

In [ ]:
def generate_response(messages):

    response = llm.invoke(messages)

    return response.content

Step 42: Extract Sources

In [ ]:
def extract_sources(documents):

    sources = []

    seen = set()

    for doc in documents:

        source = Path(
            doc.metadata.get(
                "source",
                "Unknown"
            )
        ).name

        page = doc.metadata.get(
            "page",
            "N/A"
        )

        reference = f"{source} (Page {page})"

        if reference not in seen:

            seen.add(reference)

            sources.append(reference)

    return sources

Step 43: Complete RAG Function

In [ ]:
# ===========================
# Complete RAG Pipeline
# ===========================

def ask_question(question):

    messages, documents = build_messages(question)

    answer = generate_response(messages)

    sources = extract_sources(documents)

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

Step 44: Test the Pipeline

In [ ]:
result = ask_question(
    "What is guardian?"
)

print("="*80)

print("Question")

print(result["question"])

print()

print("Answer")

print(result["answer"])

print()

print("Sources")

for source in result["sources"]:
    print("-", source)

Question
What is guardian?

Answer
[{'type': 'text', 'text': 'Based on the provided documents, Guardian (Guardian Life Insurance Company of America) is a forward-thinking mutual insurance company based in New York City. It is a leading provider of life, disability, dental, and other benefits, and is committed to championing the well-being of its customers, colleagues, and communities. \n\nAdditionally, Guardian India operates as its global capability center (GCC), delivering IT and ITES services, consulting, and business solutions to Guardian Life and its affiliates.\n\nSources:\n* JD_Business Intelligence (Intern).pdf (Page 0, Page 2, Page 3)', 'extras': {'signature': 'EqoVCqcVARFNMg/6qPXnx76ZWusWRl+MI3HNcKHRv15Cz27Ik/rYmRD/bUfznKPvqm8TkXvwI4K5uR+plt/6QWVHSaNC9T/D1ll/YUxJC8LMiEje1K5cvv2zWFmcCR/FS2DJYX/VwHmJqCbdeIsWuiz9L4a0Y98hHmpD7w7VICIJpEEQcKx5+OBLGkJU+cIiLXyRjAIV3IwkG/5AI8e9OvEs+OqzwAMDFz7OZ2ZBB2W8P5zqwtTi65xuuZoiyMtWFsO53NDaJV8DERRVVpEWbg1ko6jxFQdBOafVxPeheFY2MhY4PkcBySrBooup8+g/z

Step 45: Ask Multiple Questions

In [ ]:
questions = [

    "What is the notice period?",

    "Who approves leave requests?",

    "What are employee benefits?",

    "What is the work from home policy?"
]

for question in questions:

    result = ask_question(question)

    print("="*100)

    print("Question")

    print(result["question"])

    print()

    print("Answer")

    print(result["answer"])

    print()

    print("Sources")

    for source in result["sources"]:
        print("-", source)

    print()